# 🏥 Hepatocellular Carcinoma (HCC) Detection
## 3D Medical Imaging - Pure PyTorch (No MONAI)

**Features:**
- Pure PyTorch - No dependency issues
- 3D U-Net for volumetric segmentation
- Automatic visualization & diagnostic reports
- Works on Kaggle out-of-the-box

In [ ]:
# ============================================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================================
!pip install -q nibabel SimpleITK plotly
print("✅ Dependencies installed!")

In [ ]:
# ============================================================================
# CELL 2: IMPORTS
# ============================================================================
import os
import glob
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import nibabel as nib
from scipy import ndimage
from scipy.ndimage import zoom, label as scipy_label
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Seeds
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ============================================================================
# CELL 3: CONFIGURATION
# ============================================================================
@dataclass
class Config:
    # Kaggle dataset paths
    data_paths: List[str] = field(default_factory=lambda: [
        '/kaggle/input/cbct-liver-and-liver-tumor-segmentation-train-data',
        '/kaggle/input/liver-tumor-segmentation-part-2',
        '/kaggle/input/liver-tumor-segmentation',
        '/kaggle/input/lits-liver-tumor-segmentation-challenge',
        '/kaggle/input/3d-liver-segmentation'
    ])
    
    output_dir: str = '/kaggle/working/output'
    model_dir: str = '/kaggle/working/models'
    
    # Preprocessing
    hu_min: float = -75.0
    hu_max: float = 150.0
    target_spacing: Tuple[float, float, float] = (1.5, 1.5, 2.0)
    crop_size: Tuple[int, int, int] = (96, 96, 48)
    
    # Training
    batch_size: int = 2
    num_workers: int = 2
    max_epochs: int = 50
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    
    # Split
    train_ratio: float = 0.65
    val_ratio: float = 0.10
    test_ratio: float = 0.25
    
    num_classes: int = 3
    malignancy_threshold: float = 0.10
    use_amp: bool = True
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.model_dir, exist_ok=True)

config = Config()
print("⚙️ Configuration loaded!")

In [ ]:
# ============================================================================
# CELL 4: DATA DISCOVERY
# ============================================================================

def find_nifti_pairs(data_paths: List[str]) -> List[Dict]:
    """Find all volume-segmentation pairs."""
    pairs = []
    
    for root_path in data_paths:
        root = Path(root_path)
        if not root.exists():
            continue
        
        # Pattern: volume*.nii / segmentation*.nii
        for vol in root.rglob('volume*.nii*'):
            seg = Path(str(vol).replace('volume', 'segmentation'))
            if seg.exists():
                pid = vol.stem.split('-')[-1].split('_')[-1]
                pairs.append({'image': str(vol), 'label': str(seg), 'patient_id': f'lits_{pid}'})
        
        # Pattern: liver*.nii
        for vol in root.rglob('liver*.nii*'):
            for pattern in ['label', 'seg', 'mask']:
                seg = Path(str(vol).replace('liver', pattern))
                if seg.exists():
                    pid = vol.stem.split('_')[-1]
                    pairs.append({'image': str(vol), 'label': str(seg), 'patient_id': f'liver_{pid}'})
                    break
    
    print(f"📁 Found {len(pairs)} image-label pairs")
    return pairs


def load_nifti(filepath: str):
    nii = nib.load(filepath)
    volume = nii.get_fdata().astype(np.float32)
    spacing = nii.header.get_zooms()[:3]
    return volume, spacing


def preprocess_volume(volume, spacing, target_spacing=(1.5, 1.5, 2.0), 
                      hu_min=-75, hu_max=150, is_label=False):
    # Resample
    if spacing != target_spacing:
        factors = [s/t for s, t in zip(spacing, target_spacing)]
        order = 0 if is_label else 1
        volume = zoom(volume, factors, order=order)
    
    if not is_label:
        volume = np.clip(volume, hu_min, hu_max)
        volume = (volume - hu_min) / (hu_max - hu_min)
    
    return volume


# Find data
data_pairs = find_nifti_pairs(config.data_paths)
USE_SYNTHETIC = len(data_pairs) == 0
if USE_SYNTHETIC:
    print("⚠️ No data found - using synthetic data for demo")
else:
    print(f"✅ Using real data: {len(data_pairs)} samples")

In [ ]:
# ============================================================================
# CELL 5: DATASET
# ============================================================================

class LiverDataset(Dataset):
    def __init__(self, data_pairs, config, is_train=True, use_synthetic=False):
        self.data_pairs = data_pairs
        self.config = config
        self.is_train = is_train
        self.use_synthetic = use_synthetic
        self.crop_size = config.crop_size
    
    def __len__(self):
        return 100 if self.use_synthetic else len(self.data_pairs)
    
    def _generate_synthetic(self, idx):
        np.random.seed(idx)
        D, H, W = self.crop_size
        
        # Create volume with liver and tumors
        volume = np.random.randn(D, H, W).astype(np.float32) * 0.1 + 0.3
        label = np.zeros((D, H, W), dtype=np.int64)
        
        # Liver ellipsoid
        center = (D//2, H//2, W//2)
        radii = (D//3, H//3, W//3)
        z, y, x = np.ogrid[:D, :H, :W]
        liver_mask = ((z-center[0])/radii[0])**2 + ((y-center[1])/radii[1])**2 + ((x-center[2])/radii[2])**2 <= 1
        volume[liver_mask] = np.random.randn(liver_mask.sum()).astype(np.float32) * 0.1 + 0.6
        label[liver_mask] = 1
        
        # Tumors
        for _ in range(np.random.randint(0, 4)):
            tc = (np.random.randint(D//4, 3*D//4), np.random.randint(H//4, 3*H//4), np.random.randint(W//4, 3*W//4))
            tr = np.random.randint(3, 8)
            tumor = ((z-tc[0])**2 + (y-tc[1])**2 + (x-tc[2])**2) <= tr**2
            tumor = tumor & liver_mask
            volume[tumor] = 0.8 + np.random.randn(tumor.sum()).astype(np.float32) * 0.1
            label[tumor] = 2
        
        return volume, label
    
    def _crop(self, volume, label):
        D, H, W = volume.shape
        td, th, tw = self.crop_size
        
        # Pad if needed
        if D < td: volume = np.pad(volume, ((0, td-D), (0,0), (0,0))); label = np.pad(label, ((0, td-D), (0,0), (0,0))); D = td
        if H < th: volume = np.pad(volume, ((0,0), (0, th-H), (0,0))); label = np.pad(label, ((0,0), (0, th-H), (0,0))); H = th
        if W < tw: volume = np.pad(volume, ((0,0), (0,0), (0, tw-W))); label = np.pad(label, ((0,0), (0,0), (0, tw-W))); W = tw
        
        if self.is_train:
            d = np.random.randint(0, max(1, D-td))
            h = np.random.randint(0, max(1, H-th))
            w = np.random.randint(0, max(1, W-tw))
        else:
            d, h, w = (D-td)//2, (H-th)//2, (W-tw)//2
        
        return volume[d:d+td, h:h+th, w:w+tw], label[d:d+td, h:h+th, w:w+tw]
    
    def _augment(self, volume, label):
        for axis in range(3):
            if np.random.random() < 0.5:
                volume = np.flip(volume, axis)
                label = np.flip(label, axis)
        if np.random.random() < 0.3:
            volume = volume + np.random.randn(*volume.shape).astype(np.float32) * 0.05
        return np.clip(volume.copy(), 0, 1), label.copy()
    
    def __getitem__(self, idx):
        if self.use_synthetic:
            volume, label = self._generate_synthetic(idx)
        else:
            pair = self.data_pairs[idx]
            volume, spacing = load_nifti(pair['image'])
            label, _ = load_nifti(pair['label'])
            volume = preprocess_volume(volume, spacing, self.config.target_spacing, 
                                       self.config.hu_min, self.config.hu_max, False)
            label = preprocess_volume(label, spacing, self.config.target_spacing, 
                                      is_label=True).astype(np.int64)
        
        volume, label = self._crop(volume, label)
        if self.is_train:
            volume, label = self._augment(volume, label)
        
        return {
            'image': torch.from_numpy(volume).unsqueeze(0).float(),
            'label': torch.from_numpy(label).long(),
            'idx': idx
        }

print("✅ Dataset class defined!")

In [ ]:
# ============================================================================
# CELL 6: 3D U-NET MODEL
# ============================================================================

class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch)
        )
        self.shortcut = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm3d(out_ch)
        ) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class UNet3D(nn.Module):
    def __init__(self, in_ch=1, num_classes=3, features=[32, 64, 128, 256]):
        super().__init__()
        
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        
        # Encoder
        prev = in_ch
        for f in features:
            self.encoders.append(ResBlock3D(prev, f))
            self.pools.append(nn.MaxPool3d(2))
            prev = f
        
        # Bottleneck
        self.bottleneck = ResBlock3D(features[-1], features[-1]*2)
        
        # Decoder
        rev = features[::-1]
        for i, f in enumerate(rev):
            in_f = features[-1]*2 if i == 0 else rev[i-1]
            self.upconvs.append(nn.ConvTranspose3d(in_f, f, 2, stride=2))
            self.decoders.append(ResBlock3D(f*2, f))
        
        self.output = nn.Conv3d(features[0], num_classes, 1)
        self._init_weights()
        print(f"🧠 UNet3D: {sum(p.numel() for p in self.parameters()):,} params")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)
            skips.append(x)
            x = pool(x)
        
        x = self.bottleneck(x)
        
        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        
        return self.output(x)


# Test
model = UNet3D(1, 3).to(DEVICE)
test_out = model(torch.randn(1, 1, 48, 48, 24).to(DEVICE))
print(f"✅ Test: Input (1,1,48,48,24) → Output {test_out.shape}")

In [ ]:
# ============================================================================
# CELL 7: LOSS AND METRICS
# ============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        num_classes = pred.shape[1]
        pred_soft = F.softmax(pred, dim=1)
        target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()
        
        dice_sum = 0.0
        for c in range(1, num_classes):  # Skip background
            intersection = (pred_soft[:, c] * target_onehot[:, c]).sum()
            union = pred_soft[:, c].sum() + target_onehot[:, c].sum()
            dice_sum += (2 * intersection + self.smooth) / (union + self.smooth)
        
        return 1 - dice_sum / (num_classes - 1)


class HybridLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.dice = DiceLoss()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
    
    def forward(self, pred, target):
        return 0.5 * self.dice(pred, target) + 0.5 * self.ce(pred, target)


def compute_metrics(pred, target, num_classes=3):
    scores = {}
    names = ['bg', 'liver', 'tumor']
    
    for c in range(num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        inter = (p * t).sum()
        union = p.sum() + t.sum()
        scores[f'dice_{names[c]}'] = ((2 * inter / union) if union > 0 else 1.0).item()
    
    scores['dice_mean'] = np.mean([scores['dice_liver'], scores['dice_tumor']])
    
    # NPV for tumor
    tumor_pred = (pred == 2).float()
    tumor_target = (target == 2).float()
    tn = ((tumor_pred == 0) & (tumor_target == 0)).sum().float()
    fn = ((tumor_pred == 0) & (tumor_target == 1)).sum().float()
    scores['npv'] = (tn / (tn + fn)).item() if (tn + fn) > 0 else 1.0
    
    return scores

print("✅ Loss and metrics defined!")

In [ ]:
# ============================================================================
# CELL 8: TRAINER
# ============================================================================

class Trainer:
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        weights = torch.tensor([1.0, 2.0, 5.0]).to(DEVICE)
        self.criterion = HybridLoss(class_weights=weights)
        self.optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        self.scheduler = CosineAnnealingWarmRestarts(self.optimizer, T_0=10, T_mult=2)
        self.scaler = GradScaler() if config.use_amp else None
        
        self.history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'val_npv': [], 'lr': []}
        self.best_dice = 0.0
    
    def train_epoch(self, epoch):
        self.model.train()
        total_loss, total_dice = 0.0, 0.0
        
        for batch_idx, batch in enumerate(self.train_loader):
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            self.optimizer.zero_grad()
            
            if self.config.use_amp:
                with autocast():
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
            
            total_loss += loss.item()
            pred = torch.argmax(outputs, dim=1)
            total_dice += compute_metrics(pred, labels)['dice_mean']
        
        return total_loss / len(self.train_loader), total_dice / len(self.train_loader)
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss, total_dice, total_npv = 0.0, 0.0, 0.0
        
        for batch in self.val_loader:
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            total_loss += loss.item()
            pred = torch.argmax(outputs, dim=1)
            metrics = compute_metrics(pred, labels)
            total_dice += metrics['dice_mean']
            total_npv += metrics['npv']
        
        n = len(self.val_loader)
        return total_loss/n, total_dice/n, total_npv/n
    
    def train(self, epochs):
        print("\n" + "="*50)
        print("🚀 STARTING TRAINING")
        print("="*50)
        
        for epoch in range(1, epochs + 1):
            train_loss, train_dice = self.train_epoch(epoch)
            val_loss, val_dice, val_npv = self.validate()
            self.scheduler.step()
            lr = self.optimizer.param_groups[0]['lr']
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_dice'].append(train_dice)
            self.history['val_dice'].append(val_dice)
            self.history['val_npv'].append(val_npv)
            self.history['lr'].append(lr)
            
            is_best = val_dice > self.best_dice
            if is_best:
                self.best_dice = val_dice
                torch.save(self.model.state_dict(), f"{self.config.model_dir}/best_model.pth")
            
            print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f}/{val_loss:.4f} | Dice: {train_dice:.4f}/{val_dice:.4f} | NPV: {val_npv:.4f} {'⭐' if is_best else ''}")
        
        print(f"\n✅ Training Complete! Best Dice: {self.best_dice:.4f}")
        return self.history

print("✅ Trainer defined!")

In [ ]:
# ============================================================================
# CELL 9: VISUALIZATION
# ============================================================================

def plot_history(history):
    """Plot training curves."""
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig = make_subplots(rows=2, cols=2, subplot_titles=('Loss', 'Dice Score', 'NPV', 'Learning Rate'))
    
    fig.add_trace(go.Scatter(x=list(epochs), y=history['train_loss'], name='Train Loss', line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=list(epochs), y=history['val_loss'], name='Val Loss', line=dict(color='red')), row=1, col=1)
    fig.add_trace(go.Scatter(x=list(epochs), y=history['train_dice'], name='Train Dice', line=dict(color='blue')), row=1, col=2)
    fig.add_trace(go.Scatter(x=list(epochs), y=history['val_dice'], name='Val Dice', line=dict(color='red')), row=1, col=2)
    fig.add_trace(go.Scatter(x=list(epochs), y=history['val_npv'], name='Val NPV', line=dict(color='green')), row=2, col=1)
    fig.add_trace(go.Scatter(x=list(epochs), y=history['lr'], name='LR', line=dict(color='purple')), row=2, col=2)
    
    fig.update_layout(height=600, title_text="Training History", showlegend=True)
    fig.show()


def visualize_slice(image, label, pred, slice_idx=None):
    """Visualize a prediction slice."""
    if slice_idx is None:
        slice_idx = image.shape[0] // 2
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(image[slice_idx], cmap='gray')
    axes[0].set_title('CT Image')
    axes[0].axis('off')
    
    axes[1].imshow(label[slice_idx], cmap='viridis', vmin=0, vmax=2)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    
    axes[2].imshow(pred[slice_idx], cmap='viridis', vmin=0, vmax=2)
    axes[2].set_title('Prediction')
    axes[2].axis('off')
    
    overlay = image[slice_idx].copy()
    axes[3].imshow(overlay, cmap='gray')
    mask = np.ma.masked_where(pred[slice_idx] == 0, pred[slice_idx])
    axes[3].imshow(mask, cmap='hot', alpha=0.5, vmin=0, vmax=2)
    axes[3].set_title('Overlay')
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()


def visualize_3d(segmentation, title="3D View"):
    """3D visualization."""
    tumor = np.where(segmentation == 2)
    liver = np.where(segmentation == 1)
    
    fig = go.Figure()
    
    if len(liver[0]) > 0:
        step = max(1, len(liver[0]) // 3000)
        fig.add_trace(go.Scatter3d(
            x=liver[0][::step], y=liver[1][::step], z=liver[2][::step],
            mode='markers', marker=dict(size=2, color='blue', opacity=0.3), name='Liver'
        ))
    
    if len(tumor[0]) > 0:
        step = max(1, len(tumor[0]) // 3000)
        fig.add_trace(go.Scatter3d(
            x=tumor[0][::step], y=tumor[1][::step], z=tumor[2][::step],
            mode='markers', marker=dict(size=4, color='red', opacity=0.8), name='Tumor'
        ))
    
    fig.update_layout(title=title, height=500,
                      scene=dict(xaxis_title='D', yaxis_title='H', zaxis_title='W'))
    fig.show()


def diagnostic_report(pred, config):
    """Generate diagnostic report with 10% threshold."""
    tumor_mask = (pred == 2).astype(np.int32)
    labeled, num = scipy_label(tumor_mask)
    
    voxel_vol = np.prod(config.target_spacing)
    lesions = []
    
    for i in range(1, num + 1):
        mask = (labeled == i)
        count = mask.sum()
        vol = count * voxel_vol
        if vol < 10: continue
        
        coords = np.where(mask)
        diameter = max(c.max() - c.min() for c in coords) * config.target_spacing[0]
        lesions.append({'id': i, 'volume_cc': vol/1000, 'diameter_mm': diameter})
    
    tumor_vol = sum(l['volume_cc'] for l in lesions)
    liver_vol = (pred == 1).sum() * voxel_vol / 1000
    burden = tumor_vol / (liver_vol + 1e-8)
    cancer = burden >= config.malignancy_threshold
    
    return {
        'cancer_detected': cancer,
        'num_lesions': len(lesions),
        'tumor_volume_cc': tumor_vol,
        'liver_volume_cc': liver_vol,
        'tumor_burden_pct': burden * 100,
        'lesions': lesions
    }


def print_report(report):
    status = "🔴 CANCER DETECTED" if report['cancer_detected'] else "🟢 NO CANCER"
    print(f"\n{'='*50}")
    print(f"🏥 DIAGNOSTIC REPORT")
    print(f"{'='*50}")
    print(f"Status: {status}")
    print(f"Lesions: {report['num_lesions']}")
    print(f"Tumor Volume: {report['tumor_volume_cc']:.2f} cc")
    print(f"Liver Volume: {report['liver_volume_cc']:.2f} cc")
    print(f"Tumor Burden: {report['tumor_burden_pct']:.2f}%")
    for l in report['lesions']:
        print(f"  Lesion #{l['id']}: {l['volume_cc']:.3f} cc, {l['diameter_mm']:.1f} mm")
    print(f"{'='*50}")

print("✅ Visualization functions defined!")

In [ ]:
# ============================================================================
# CELL 10: PREPARE DATA AND TRAIN
# ============================================================================

# Create datasets
train_ds = LiverDataset(data_pairs, config, is_train=True, use_synthetic=USE_SYNTHETIC)
val_ds = LiverDataset(data_pairs, config, is_train=False, use_synthetic=USE_SYNTHETIC)
test_ds = LiverDataset(data_pairs, config, is_train=False, use_synthetic=USE_SYNTHETIC)

print(f"📦 Datasets: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

print(f"🔄 Loaders: Train={len(train_loader)} batches, Val={len(val_loader)} batches")

In [ ]:
# ============================================================================
# CELL 11: TRAIN MODEL
# ============================================================================

# Initialize
model = UNet3D(1, 3).to(DEVICE)
trainer = Trainer(model, train_loader, val_loader, config)

# Train
EPOCHS = 30  # Adjust as needed
history = trainer.train(EPOCHS)

In [ ]:
# ============================================================================
# CELL 12: VISUALIZE TRAINING
# ============================================================================

plot_history(history)

print(f"\n📊 Final Results:")
print(f"   Best Dice: {max(history['val_dice']):.4f}")
print(f"   Best NPV: {max(history['val_npv']):.4f}")

In [ ]:
# ============================================================================
# CELL 13: TEST EVALUATION
# ============================================================================

# Load best model
try:
    model.load_state_dict(torch.load(f"{config.model_dir}/best_model.pth"))
    print("✅ Loaded best model")
except:
    print("⚠️ Using current model")

model.eval()
predictions = []
test_scores = []

print("\n🔍 Evaluating test set...")

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        images = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        
        outputs = model(images)
        pred = torch.argmax(outputs, dim=1)
        
        metrics = compute_metrics(pred, labels)
        test_scores.append(metrics)
        
        predictions.append({
            'image': images[0, 0].cpu().numpy(),
            'label': labels[0].cpu().numpy(),
            'pred': pred[0].cpu().numpy(),
            'metrics': metrics
        })
        
        if i < 5:
            print(f"   Sample {i+1}: Dice={metrics['dice_mean']:.4f}, NPV={metrics['npv']:.4f}")

mean_dice = np.mean([s['dice_mean'] for s in test_scores])
mean_npv = np.mean([s['npv'] for s in test_scores])

print(f"\n📊 Test Results:")
print(f"   Mean Dice: {mean_dice:.4f}")
print(f"   Mean NPV: {mean_npv:.4f}")

In [ ]:
# ============================================================================
# CELL 14: VISUALIZE PREDICTIONS
# ============================================================================

print("\n🖼️ Sample Predictions:")

for i, p in enumerate(predictions[:5]):
    print(f"\n--- Sample {i+1} (Dice: {p['metrics']['dice_mean']:.4f}) ---")
    visualize_slice(p['image'], p['label'], p['pred'])
    
    report = diagnostic_report(p['pred'], config)
    print_report(report)

In [ ]:
# ============================================================================
# CELL 15: 3D VISUALIZATION
# ============================================================================

if predictions:
    best_idx = np.argmax([p['metrics']['dice_mean'] for p in predictions])
    best = predictions[best_idx]
    
    print(f"\n🎯 Best Prediction (Dice: {best['metrics']['dice_mean']:.4f})")
    
    visualize_3d(best['label'], "Ground Truth")
    visualize_3d(best['pred'], "Prediction")

In [ ]:
# ============================================================================
# CELL 16: SUMMARY
# ============================================================================

print(f"""
{'='*60}
📋 HCC DETECTION PIPELINE SUMMARY
{'='*60}

🏗️ Architecture:
   • Model: 3D U-Net (ResBlocks)
   • Input: {config.crop_size}
   • Classes: Background, Liver, Tumor
   • Parameters: {sum(p.numel() for p in model.parameters()):,}

📊 Preprocessing:
   • HU Window: [{config.hu_min}, {config.hu_max}]
   • Spacing: {config.target_spacing} mm

🎯 Training:
   • Loss: Dice + CrossEntropy
   • Optimizer: Adam (lr={config.learning_rate})
   • Epochs: {EPOCHS}
   • Best Val Dice: {max(history['val_dice']):.4f}

🔬 Test Results:
   • Mean Dice: {mean_dice:.4f}
   • Mean NPV: {mean_npv:.4f}

🏥 Diagnostic:
   • Malignancy Threshold: {config.malignancy_threshold*100}%

{'='*60}
✅ Pipeline Complete!
{'='*60}
""")

# Save results
results = {
    'test_dice': mean_dice,
    'test_npv': mean_npv,
    'best_val_dice': max(history['val_dice']),
    'history': history
}
with open(f"{config.output_dir}/results.json", 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"💾 Results saved to {config.output_dir}/results.json")

## ✅ Done!

This notebook:
- Uses **pure PyTorch** (no MONAI)
- Works on **Kaggle** out-of-the-box
- Includes **visualizations** (2D slices, 3D views, training curves)
- Generates **diagnostic reports** with 10% malignancy threshold
- Falls back to **synthetic data** if datasets not available

To use with real data:
1. Add liver tumor datasets to Kaggle
2. Update `data_paths` in Config
3. Run all cells